# Hype Score Calculation

Calculate hype scores for historical games with adjustable parameters.

Current rating snapshots are not historical pregame evidence. The scoring functions leave qualification unavailable for past or started games. Do not interpret these comparisons as validation of historical playoff stakes, or use them to tune those stakes. Strength and championship inputs also lack a historical evidence guarantee. The evaluation time is explicit, and team lookups include the game season.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hype_score.database import get_engine, load_games, load_team_ratings_for_games, load_pregame_odds
from hype_score.config import HypeConfig
from hype_score.scoring import calculate_hype_scores_batch

## Load Data

In [ ]:
evaluated_at = pd.Timestamp.now(tz="UTC")
engine = get_engine()

games = load_games(engine, status='final', limit=1000)
print(f"Loaded {len(games):,} completed games")

pregame_odds = load_pregame_odds(engine, game_ids=games['id'].tolist())
print(f"Loaded pregame odds for {len(pregame_odds):,} games")

ratings = load_team_ratings_for_games(engine, games, evaluated_at=evaluated_at)
print(f"Loaded {len(ratings):,} team ratings")

## Calculate Hype Scores with Default Config

In [ ]:
config = HypeConfig.default()


print("Sport weights:")
for sport, weights in config.sport_weights.items():
    print(f"  {sport}: stakes={weights.stakes_weight:.2f}, quality={weights.quality_weight:.2f}, closeness={weights.closeness_weight:.2f}")

In [ ]:
hype_results = calculate_hype_scores_batch(games, pregame_odds, ratings, config, evaluated_at=evaluated_at)
print(f"Calculated hype scores for {len(hype_results):,} games")
hype_results.head(10)

## Hype Score Distribution

In [ ]:
fig = px.histogram(
    hype_results, 
    x='hype_score', 
    color='sport',
    nbins=50,
    title='Hype Score Distribution by Sport'
)
fig.add_vline(x=40, line_dash='dash', annotation_text='Low/Med threshold')
fig.add_vline(x=70, line_dash='dash', annotation_text='Med/High threshold')
fig.show()

In [ ]:
hype_results.groupby('sport')['hype_score'].describe().round(1)

## Component Breakdown

In [ ]:
components = hype_results[['stakes_score', 'quality_score', 'closeness_score', 'hype_score']]
print("Component correlations:")
components.corr().round(2)

In [ ]:
fig = px.scatter_matrix(
    hype_results,
    dimensions=['stakes_score', 'quality_score', 'closeness_score', 'hype_score'],
    color='sport',
    title='Component Relationships'
)
fig.update_layout(height=800)
fig.show()

## Top Hype Games

In [ ]:
top_hype = hype_results.nlargest(20, 'hype_score')
top_hype[['home_team', 'away_team', 'sport', 'hype_score', 'stakes_score', 'quality_score', 'closeness_score', 'scheduled_start']]

In [ ]:
low_hype = hype_results.nsmallest(20, 'hype_score')
low_hype[['home_team', 'away_team', 'sport', 'hype_score', 'stakes_score', 'quality_score', 'closeness_score', 'scheduled_start']]

## Interactive Parameter Tuning

Use sliders to adjust weights and see results update in real-time.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

stakes_slider = widgets.FloatSlider(value=0.45, min=0, max=1, step=0.05, description='Stakes:')
quality_slider = widgets.FloatSlider(value=0.35, min=0, max=1, step=0.05, description='Quality:')
closeness_slider = widgets.FloatSlider(value=0.20, min=0, max=1, step=0.05, description='Closeness:')

output = widgets.Output()

def update_scores(change):
    with output:
        clear_output(wait=True)
        
        total = stakes_slider.value + quality_slider.value + closeness_slider.value
        if total == 0:
            print("Weights must not all be zero")
            return
        
        stakes_w = stakes_slider.value / total
        quality_w = quality_slider.value / total
        closeness_w = closeness_slider.value / total
        
        print(f"Normalized weights: stakes={stakes_w:.2f}, quality={quality_w:.2f}, closeness={closeness_w:.2f}")
        
        from hype_score.config import SportWeights
        custom_config = HypeConfig.default()
        custom_config.sport_weights['nfl'] = SportWeights(stakes_w, quality_w, closeness_w)
        
        nfl_games = games[games['sport'] == 'nfl']
        if len(nfl_games) > 0:
            new_results = calculate_hype_scores_batch(nfl_games, pregame_odds, ratings, custom_config, evaluated_at=evaluated_at)
            print(f"\nNFL Hype Score Stats:")
            print(new_results['hype_score'].describe())

stakes_slider.observe(update_scores, names='value')
quality_slider.observe(update_scores, names='value')
closeness_slider.observe(update_scores, names='value')

display(widgets.VBox([stakes_slider, quality_slider, closeness_slider, output]))
update_scores(None)   